# Experiment 6.0 — Multi-timescale phase-aware evidence + synapse-update ablation

Analysis-only notebook. All 54 SNN trainings, checkpoint selection, test evaluation, activity extraction, and aggregation are completed by the Exp6.0 CPU pipeline before this notebook is opened.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

REPO_ROOT = Path('..').resolve()
ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_6_0_multiscale_phase_evidence' / 'single_hidden_multiscale_phase_evidence_v1'
runs = pd.read_csv(ROOT / 'runs.csv')
summary = pd.read_csv(ROOT / 'summary.csv')
paired = pd.read_csv(ROOT / 'paired_synapse_update_deltas.csv')
history_index = pd.read_csv(ROOT / 'history_index.csv')
baseline = json.loads((ROOT / 'baseline_fixed250_linear.json').read_text())
manifest = json.loads((ROOT / 'manifest.json').read_text())
sample = json.loads((ROOT / 'visualization_sample.json').read_text())
print('runs:', len(runs))
print('synapse modes:', runs.synapse_mode.value_counts().to_dict())
print('visualization sample:', sample)


## Run-level results and Fixed250 reference

In [ ]:
display(runs.sort_values(['synapse_mode', 'objective', 'mem_shift', 'seed']))
display(summary)
baseline_metrics = baseline['baseline']['metrics']
display(pd.DataFrame(baseline_metrics).T)


## Test Balanced Accuracy vs hidden membrane shift

In [ ]:
objective_labels = {
    'whole_count': 'WholeCount',
    'whole_count_hce': 'WholeCount + HCE',
    'whole_count_contextual_gain': 'WholeCount + Contextual Gain',
}
mode_labels = {
    'normalized_unit_dc': 'Normalized: alpha I + (1-alpha) Wx',
    'legacy_unnormalized': 'Legacy: alpha I + Wx',
}
reference = float(baseline_metrics['test']['balanced_accuracy'])
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (mode, mode_label) in zip(axes, mode_labels.items()):
    mode_rows = runs[runs.synapse_mode == mode]
    for objective, label in objective_labels.items():
        subset = mode_rows[mode_rows.objective == objective]
        stats = subset.groupby('mem_shift').test_ba.agg(['mean', 'std']).reindex([1, 2, 3])
        ax.errorbar(stats.index, stats['mean'], yerr=stats['std'].fillna(0), marker='o', capsize=4, label=label)
    ax.axhline(reference, linestyle='--', label=f'Raw Fixed250 + Linear ({reference:.3f})')
    ax.set_title(mode_label)
    ax.set_xticks([1, 2, 3])
    ax.set_xlabel('Hidden membrane shift')
    ax.set_ylim(0, 1)
axes[0].set_ylabel('Test Balanced Accuracy')
axes[-1].legend()
plt.show()


## Paired synapse-update effect

In [ ]:
display(paired.sort_values(['objective', 'mem_shift', 'seed']))
delta_col = 'test_ba_legacy_minus_normalized'
fig, ax = plt.subplots(figsize=(8, 5))
for objective, label in objective_labels.items():
    subset = paired[paired.objective == objective]
    stats = subset.groupby('mem_shift')[delta_col].agg(['mean', 'std']).reindex([1, 2, 3])
    ax.errorbar(stats.index, stats['mean'], yerr=stats['std'].fillna(0), marker='o', capsize=4, label=label)
ax.axhline(0, linestyle='--')
ax.set_xticks([1, 2, 3])
ax.set_xlabel('Hidden membrane shift')
ax.set_ylabel('Test BA: legacy - normalized')
ax.legend()
plt.show()


## Validation BA and hidden firing diagnostics

In [ ]:
fr_cols = ['hidden_s4_fr_valid', 'hidden_s5_fr_valid', 'hidden_s6_fr_valid', 'hidden_s4_fr_tail', 'hidden_s5_fr_tail', 'hidden_s6_fr_tail']
display(runs.groupby(['synapse_mode', 'objective', 'mem_shift'])[['val_ba', *fr_cols]].agg(['mean', 'std']))


## Matched learning curves and spike rasters

All runs use the same deterministic validation segment. The following cell shows one matched normalized/legacy pair.

In [ ]:
pair_rows = history_index[(history_index.objective == 'whole_count') & (history_index.mem_shift == 1) & (history_index.seed == 11)].sort_values('synapse_mode')
for _, row in pair_rows.iterrows():
    print(row.synapse_mode)
    display(Image(filename=str(REPO_ROOT / row.learning_curve_png)))
    display(Image(filename=str(REPO_ROOT / row.raster_png)))


## HCE original-order vs reversed-order rasters

In [ ]:
hce_rows = history_index[(history_index.objective == 'whole_count_hce') & (history_index.mem_shift == 1) & (history_index.seed == 11)].sort_values('synapse_mode')
for _, row in hce_rows.iterrows():
    original = REPO_ROOT / row.raster_png
    reversed_png = original.with_name(original.stem + '__reversed.png')
    print(row.synapse_mode)
    display(Image(filename=str(original)))
    display(Image(filename=str(reversed_png)))


## Interpretation checklist

1. At matched objective, membrane shift, and seed, how does removing `(1-alpha)` change BA?
2. Does the legacy update increase s5/s6 firing or post-end tail persistence as expected from its larger DC gain?
3. Is any legacy performance advantage accompanied by saturation/persistent activity rather than cleaner phase-aware evidence?
4. Does HCE or Contextual Gain interact differently with the two synaptic-current equations?
5. Which condition gets closest to Raw Fixed250 + Linear while preserving interpretable hidden activity?